# Problem 3 — Phase 4: Kernel pilot (not full tree)

**Index:** [`hierarchical_problem3_index.ipynb`](hierarchical_problem3_index.ipynb) · **Prev:** [Phase 3](hierarchical_problem3_phase3_by_depth.ipynb)

[`pilot_kernel_svc_on_edge`](hierarchical_summary_metrics.py): subsampled **train**, hold-out F1 + fit time for **linear / RBF / poly** `SVC` on **one** Root→H1 edge. Does not scale to the full taxonomy.


In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import pandas as pd
from IPython.display import display

from hierarchical_training_data import make_multilabel_binary_pool_data
from topic_hierarchy import load_topic_tree, summary


def homework4_base() -> Path:
    here = Path.cwd().resolve()
    if (here / "topics.csv").exists():
        return here
    for p in [here, *here.parents]:
        if (p / "topics.csv").exists():
            return p
    raise FileNotFoundError("topics.csv not found — cwd should be Homework 4")


BASE = homework4_base()
tree = load_topic_tree(str(BASE / "topics.csv"))
mldata = make_multilabel_binary_pool_data(base_path=str(BASE))
SPLIT = "test"
RESTRICT = True
MAX_FEATURES = 8000

print("BASE", BASE)
print(summary(tree))
print("train", len(mldata.train_ids()), "test", len(mldata.test_ids()))

from hierarchical_summary_metrics import pilot_kernel_svc_on_edge


In [ ]:
root = tree.traversal_root
children = tree.children.get(root, [])
pilot_edge = None
for ch in children:
    Xtr, ytr = mldata.binary_edge_dataset(root, ch, "train")
    if len(set(ytr)) >= 2 and len(Xtr) >= 100:
        pilot_edge = (root, ch)
        break
if pilot_edge:
    print("Pilot edge", pilot_edge)
    pilot_res = pilot_kernel_svc_on_edge(mldata, pilot_edge[0], pilot_edge[1])
    display(pd.DataFrame(pilot_res).T)
else:
    print("No suitable H1 edge for pilot")


### Notes

If linear ≈ nonlinear F1 but kernels are much slower per fit, prefer **linear `LinearSVC`** for full-tree training. Full-tree kernel SVMs are usually impractical here.
